# 20.2 PyTorch'un yapı taşları

*Makine Öğrenmesi Çalışma Rehberi · Bölüm 20: PyTorch ile Uygulamalı Örnekler*

**Nasıl kullanılır?** Hücreleri sırayla çalıştır (Shift + Enter) ya da menüden *Çalışma zamanı → Tümünü çalıştır*'ı seç. GPU gerekmez. Kodu değiştirip yeniden çalıştırmaktan çekinme. Köşeli parantezli numaralar ([76] gibi) PDF'teki kaynakçaya karşılık gelir.

**Tablo 20.1: PyTorch'un temel parçaları**

| Yapı taşı | Ne işe yarar? | Rehberdeki karşılığı |
|---|---|---|
| `torch.tensor` | NumPy dizisine benzeyen çok boyutlu dizi; GPU'da da çalışabilir | — |
| Otomatik türev (`backward()`) | Kaybın tüm ağırlıklara göre türevini zincir kuralıyla otomatik hesaplar | Geri yayılım (14.4) |
| `nn.Module`, `nn.Sequential` | Modelin tanımı: katmanlar ve ileri yayılım (forward) | Yapay sinir ağı (14.3) |
| `nn.MSELoss`, `nn.CrossEntropyLoss` | Regresyon ve sınıflandırma için kayıp fonksiyonları | 3.2 ve 5.4 |
| `torch.optim` (`SGD`, `Adam`) | Ağırlıkları güncelleme kuralları | 4.2 ve 4.5 |
| `DataLoader` | Veriyi her epokta karıştırıp mini-yığınlara böler | 4.4 |

### Otomatik türev: Bölüm 14'teki örnek, bu kez PyTorch'la

Bölüm 14'te zincir kuralını elle uygulamıştık: x = 2, y = 10 ve w = 3 için kayıp 16, türev −16 ve güncellenmiş ağırlık 3,8 çıkmıştı. PyTorch aynı hesabı şöyle yapar:

In [ ]:
"""Bölüm 14'teki zincir kuralı örneğini PyTorch'un otomatik türeviyle tekrar hesaplar."""
import torch

w = torch.tensor(3.0, requires_grad=True)  # öğrenilecek ağırlık
x, y = torch.tensor(2.0), torch.tensor(10.0)

kayip = (w * x - y) ** 2   # ileri yayılım: (3·2 − 10)² = 16
kayip.backward()           # geri yayılım: dL/dw otomatik hesaplanır
print(kayip.item(), w.grad.item())   # 16.0 -16.0

with torch.no_grad():      # güncelleme adımı türev kaydına girmesin
    w -= 0.05 * w.grad
print(round(w.item(), 4))  # 3.8

**Beklenen çıktı** (kesin)

```text
16.0 -16.0
3.8
```

`requires_grad=True`, PyTorch'a bu tensörle yapılan işlemleri kaydetmesini söyler. `backward()` bu kaydı sondan başa izleyerek zincir kuralını uygular ve sonucu `w.grad` içine yazar. Güncelleme `torch.no_grad()` içinde yapılır, çünkü ağırlığı değiştirmek modelin hesabının bir parçası değildir. Çıktı, Bölüm 14'teki elle yaptığımız hesapla birebir aynıdır.

### Eğitim döngüsü: dört adım

*Her PyTorch eğitiminin çekirdeği* (yalnızca okumak için)

```python
for xb, yb in yukleyici:              # her mini-yığın için
    opt.zero_grad()                   # 1) önceki adımın gradyanlarını sıfırla
    kayip = kayip_fn(model(xb), yb)   # 2) ileri yayılım ve kayıp
    kayip.backward()                  # 3) geri yayılım: gradyanları hesapla
    opt.step()                        # 4) ağırlıkları güncelle
```

Bu dört satır, Bölüm 14.4'teki öğrenme döngüsünün PyTorch'taki karşılığıdır ve birazdan göreceğin iki örnekte de aynen tekrarlanır. PyTorch gradyanları varsayılan olarak biriktirir; bu yüzden `opt.zero_grad()` unutulursa her adımın gradyanı öncekilerin üzerine eklenir.